# ASG Airlines - Silver Layer

## Objective

The Silver layer transforms raw operational data into a clean, standardized and analytics-ready dataset.

### Transformations

- Duplicate removal
- Missing value treatment
- Passenger master resolution
- PII masking
- Duration calculation
- Overnight flight handling
- Surrogate key generation

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path
import hashlib

Read Bronze Data


In [3]:
# Paths
BRONZE = Path("../data/bronze")
SILVER = Path("../data/silver")

SILVER.mkdir(parents=True, exist_ok=True)

# Read Bronze tables
flights = pd.read_parquet(BRONZE / "flights.parquet")
passengers = pd.read_parquet(BRONZE / "passengers.parquet")
bookings = pd.read_parquet(BRONZE / "bookings.parquet")
payments = pd.read_parquet(BRONZE / "payments.parquet")

print("Bronze data loaded successfully.")

Bronze data loaded successfully.


## Transformation 1 — Remove Exact Duplicate Flights

Exact duplicate flight records represent repeated ingestion of the same operational event.

Business Rule:
- Remove only identical rows.
- Preserve conflicting flight IDs for later surrogate key generation.

In [4]:
before = len(flights)

flights = flights.drop_duplicates()

after = len(flights)

print(f"Removed {before-after} duplicate flight records.")

Removed 15 duplicate flight records.


Step 2 - Resolve Missing Airline


In [5]:
airline_map = {
    "AI":"Air India",
    "SJ":"SpiceJet",
    "UK":"Vistara",
    "6F":"Indigo"
}

# Extract prefix
flights["prefix"] = flights["flight_id"].str[:2]

# Fill only missing airlines
flights["airline"] = flights["airline"].fillna(
    flights["prefix"].map(airline_map)
)

flights.drop(columns="prefix", inplace=True)

STEP 3 — Booking Status

In [6]:
bookings["status"] = bookings["status"].fillna("UNKNOWN")

STEP 4 — Passenger Last Name

In [7]:
passengers["last_name"] = (
    passengers["last_name"]
    .fillna("Unknown")
)

STEP 5 — Resolve Duplicate Passenger IDs

## Transformation 5 — Master Data Resolution

Duplicate passenger IDs contain conflicting demographic information.

Resolution Strategy:

1. Calculate completeness score.
2. Keep the most complete record.
3. If tied, retain first occurrence.

In [10]:
# Completeness score
passengers["completeness_score"] = passengers.notna().sum(axis=1)

# Keep best record
passengers = (
    passengers
    .sort_values("completeness_score", ascending=False)
    .drop_duplicates("passenger_id", keep="first")
    .drop(columns="completeness_score")
)

print("Passenger master resolved.")

Passenger master resolved.


Step 6 - PII Masking


Helper Function

In [11]:
def hash_value(value):

    if pd.isna(value):
        return np.nan

    return hashlib.sha256(
        str(value).encode()
    ).hexdigest()

Apply Masking

In [12]:
# Passenger table
passengers["aadhaar_hash"] = passengers["aadhaar_id"].apply(hash_value)
passengers["email_hash"] = passengers["email"].apply(hash_value)

# Keep masked phone
passengers["phone_masked"] = passengers["phone"].str.replace(
    r"\d{4}$",
    "XXXX",
    regex=True
)

# Drop sensitive columns
passengers.drop(
    columns=["aadhaar_id","email","phone"],
    inplace=True
)

# Booking table
bookings["passport_hash"] = bookings["passport_number"].apply(hash_value)

bookings["emergency_phone_masked"] = bookings[
    "emergency_contact_phone"
].str.replace(
    r"\d{4}$",
    "XXXX",
    regex=True
)

bookings.drop(
    columns=["passport_number","emergency_contact_phone"],
    inplace=True
)

Step -7 Duration Calculation


 Duration in Minutes

In [15]:


# Clean the raw duration column
flights["duration"] = (
    flights["duration"]
    .astype(str)
    .str.strip()
    .replace({"NaT": np.nan, "nan": np.nan, "None": np.nan})
)

# Convert safely to timedelta
duration_td = pd.to_timedelta(
    flights["duration"],
    errors="coerce"
)

# Create duration in minutes
flights["duration_minutes"] = (
    duration_td.dt.total_seconds() / 60
).round().astype("Int64")

# Check if any records failed conversion
invalid_duration = flights[flights["duration_minutes"].isna()]

print(f"Invalid duration records: {len(invalid_duration)}")
flights[["duration", "duration_minutes"]].head()

Invalid duration records: 1


,duration,duration_minutes
0,02:54:00,174
1,01:48:00,108
2,01:45:00,105
3,02:36:00,156
4,04:59:00,299


Step 8 - Overnight Flights 

In [16]:
flights["is_overnight"] = (
    flights["arrival_time"].dt.date >
    flights["departure_time"].dt.date
)

Step 9 - Surrogate Keys for corrupted Flight IDs

In [17]:
flights.insert(
    0,
    "flight_key",
    range(1, len(flights)+1)
)

passengers.insert(
    0,
    "passenger_key",
    range(1, len(passengers)+1)
)

Saving The Processing 

In [18]:
tables = {
    "flights":flights,
    "passengers":passengers,
    "bookings":bookings,
    "payments":payments
}

for name, df in tables.items():

    df.to_csv(
        SILVER / f"{name}.csv",
        index=False
    )

    df.to_parquet(
        SILVER / f"{name}.parquet",
        index=False
    )

print("Silver Layer Created Successfully!")

Silver Layer Created Successfully!


# Silver Layer Summary

The Silver layer transformed the raw Bronze datasets into standardized analytical tables.

### Transformations Completed

- Removed exact duplicate flight records
- Filled missing categorical values
- Resolved duplicate passenger master records
- Masked Personally Identifiable Information (PII)
- Calculated flight duration in minutes
- Created overnight flight indicator
- Generated surrogate warehouse keys

The resulting Silver datasets are clean, secure, and ready for SQL modelling and KPI generation in the Gold layer.